#  Notebook 1: Models, Prompts & Parsers – Your First LLM Interaction

**🎯 Goal:** Learn how to interact with Large Language Models (LLMs), design effective prompts, and parse the output into structured data.

## 🧩 Concept Overview

Think of an LLM as a brilliant, but very literal, intern. 
- **Models:** The intern's brain (e.g., GPT-4, Llama3). You can choose different brains for different tasks.
- **Prompts:** The instructions you give the intern. The more specific your instructions, the better the result.
- **Parsers:** A set of rules to make the intern's messy notes into a clean, structured report (like a spreadsheet or JSON).

## 🖼️ Visual Diagram

Here’s the basic flow of what we're building today:

```
               +-----------------+      +-----------------+      +-----------------+
Your Input ===> |  PromptTemplate | ===> |       LLM       | ===> |  Output Parser  | ===> Structured Data
 (e.g., 'GPU') | (Formats Input) |      |  (Processes)    |      | (Extracts Info) |      (e.g., JSON)
               +-----------------+      +-----------------+      +-----------------+
```

## ⚙️ Setup

First, let's load our API keys from the `.env` file we created.

In [ ]:
# 🪄 Load environment variables
from dotenv import load_dotenv
import os

load_dotenv()

# Optional: Check if the API key is loaded correctly
# print(f"OpenAI API Key Loaded: {os.getenv('OPENAI_API_KEY') is not None}")

## 1. Models: The Brain of the Operation

LangChain makes it easy to switch between different LLMs. We'll primarily use OpenAI, but the same interface works for others like Hugging Face (for open-source models) or local models via Ollama.

In [ ]:
# 🤖 We'll use ChatOpenAI which is the recommended model type
from langchain_openai import ChatOpenAI

# Initialize the model. We'll use a lower temperature for more predictable output.
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.3)

# --- Other Model Examples (Optional) ---

# from langchain_community.llms import HuggingFaceEndpoint
# llm_hf = HuggingFaceEndpoint(
#     repo_id="mistralai/Mistral-7B-Instruct-v0.2", 
#     huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN")
# )

# from langchain_community.chat_models import ChatOllama
# llm_local = ChatOllama(model="llama3")

print("LLM Initialized:", llm.model_name)

## 2. Prompts: Giving Clear Instructions

A `PromptTemplate` is like a fill-in-the-blanks form. It lets you create reusable prompts and easily insert variables.

In [ ]:
from langchain_core.prompts import PromptTemplate

# 📝 Create a template. The {product} part is a variable.
prompt_template = PromptTemplate.from_template(
    "Tell me a short, funny tagline for a company that sells {product}."
)

# 🚀 Format the prompt with a specific product
formatted_prompt = prompt_template.format(product="AI-powered toasters")

print(formatted_prompt)

### Chaining It Together with LCEL

LangChain Expression Language (LCEL) is the modern way to connect components. The `|` (pipe) operator is used to chain things together, just like in a shell script.

In [ ]:
# 🔗 Chain the prompt template and the LLM together
chain = prompt_template | llm

# 🚀 Run the chain
response = chain.invoke({"product": "robotic dogs"})

print(response.content)

## 3. Parsers: Getting Structured Output

LLMs return text, but often we need data in a specific format like JSON. `OutputParsers` help us do this reliably.

We'll use `PydanticOutputParser`, which lets you define your desired data structure using a Python class. It's robust and gives you type-hinting.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

# 📊 Define the desired data structure
class ProductInfo(BaseModel):
    """Information about a product."""
    name: str = Field(description="The name of the product.")
    features: List[str] = Field(description="A list of key features.")
    price: float = Field(description="The price of the product.")

# 🔨 Create a parser instance
pydantic_parser = PydanticOutputParser(pydantic_object=ProductInfo)

# 📝 Get the format instructions that we'll inject into the prompt
format_instructions = pydantic_parser.get_format_instructions()

print(format_instructions)

### Combining Parser, Prompt, and Model

In [ ]:
# The product description we want to extract info from
product_description = """
The Galactic Zoomer 9000 is the latest innovation in personal mobility, priced at $25000. 
With its anti-gravity propulsion system and sleek, aerodynamic design, it offers a smooth ride. 
The built-in AI navigation system, 'Nova', provides real-time traffic updates. 
It also has a holographic display and omni-directional obstacle avoidance sensors.
"""

# 📝 Create a new prompt template that includes the format instructions
parser_prompt = PromptTemplate(
    template="""
    You are an expert at extracting information from product descriptions.
    Answer the user's query using the following product description:
    \n{product_description}\n
    Format your response as a JSON object according to the following instructions:
    \n{format_instructions}\n
    """,
    input_variables=["product_description"],
    partial_variables={"format_instructions": format_instructions}
)

# 🔗 Create the full chain
parser_chain = parser_prompt | llm | pydantic_parser

# 🚀 Run the chain
structured_response = parser_chain.invoke({"product_description": product_description})

print(type(structured_response))
print(structured_response)

## 🔧 Hands-On Exercise

**Goal:** Use the concepts learned to extract information from the `product-info.txt` file in the `data` directory.

1. Read the content of `data/product-info.txt`.
2. Use the `parser_chain` we just created to invoke the model with this new content.
3. Print the extracted product name and its price.

In [ ]:
# Your code here!

# 1. Read the file content
with open('../data/product-info.txt', 'r') as f:
    exercise_text = f.read()

# 2. Invoke the chain
exercise_result = parser_chain.invoke({"product_description": exercise_text})

# 3. Print the name and price
print(f"Product Name: {exercise_result.name}")
print(f"Price: ${exercise_result.price:.2f}")

## 🐞 Bug Bounty

The code below has a bug. It's trying to use a `JsonOutputParser` without giving the LLM proper instructions. Can you fix it?

**Hint:** The LLM doesn't magically know it needs to output JSON. You have to tell it!

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

# --- BROKEN CODE ---
buggy_prompt = PromptTemplate.from_template(
    "Extract the name and price from this text: {text}."
)
buggy_chain = buggy_prompt | llm | JsonOutputParser()
# try:
#     result = buggy_chain.invoke({"text": product_description})
#     print(result)
# except Exception as e:
#     print(f"It failed! Error: {e}")

# --- FIXED CODE ---
fixed_prompt = PromptTemplate.from_template(
    """Extract the name and price from this text: {text}. 
    Return a JSON object with keys 'name' and 'price'."""
)

fixed_chain = fixed_prompt | llm | JsonOutputParser()
result = fixed_chain.invoke({"text": product_description})
print("Fixed code output:", result)

## 💡 Pro Tip

When using parsers, always set a lower `temperature` (e.g., 0.1 to 0.4) on your LLM. This makes the output more predictable and deterministic, which helps the parser succeed more often. High temperatures are great for creative tasks, but bad for structured data extraction.

## 🏁 Summary

Congratulations on completing your first notebook! Here are the key takeaways:

1.  **Models are interchangeable:** LangChain provides a standard interface (`invoke`) to run different LLMs.
2.  **Prompts are powerful:** `PromptTemplate` is essential for creating dynamic and reusable instructions for the LLM.
3.  **Parsers create structure:** `OutputParser` (especially Pydantic) is the key to turning natural language responses into usable data objects in your application.